# UPAR 2027 Track 2 — CSAR pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/UPAR-Challenge-2027/blob/main/notebooks/pipeline.ipynb)

Fine-tunes every candidate configuration, scores them with one shared yardstick, runs leave-one-domain-out (LODO) for the winner, builds an ensemble and a WiSE-FT variant, exports the best one and runs inference through the exported `run.py` like the Codabench ingestion program.

All settings come from `configs/` (Hydra / OmegaConf): `configs/config.yaml` for the values, `configs/experiment/*.yaml` for the candidates. The notebook only selects a profile and adds overrides.

* `PROFILE = "smoke"`: synthetic data (real labels, random images), CPU, a few minutes.
* `PROFILE = "full"`: real images (downloaded if missing), GPU (Colab: *Runtime → Change runtime type → GPU*).

The default backbone is DINOv3 ViT-B/16 when its gated weights are reachable (licence accepted on Hugging Face and `HF_TOKEN` set, e.g. as a Colab secret) and DINOv2 ViT-B/14 otherwise.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hashirama21/UPAR-Challenge-2027.git"
IN_COLAB = "google.colab" in sys.modules

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src" / "train.py").is_file():
        ROOT = candidate
        break
else:
    ROOT = Path("/content/UPAR-Challenge-2027") if IN_COLAB else Path.cwd() / "UPAR-Challenge-2027"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hydra-core", "omegaconf"], check=True)
    from google.colab import userdata
    try:
        os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
    except Exception:
        pass
print("repository root:", ROOT)

## 1. Configuration

In [ ]:
import time

import numpy as np
from omegaconf import OmegaConf

from src.config import compose, with_values

PROFILE = "smoke"   # "smoke" | "full"
OVERRIDES = []      # extra Hydra overrides, e.g. ["optim.epochs=10", "experiment.lodo=all"]

cfg = compose([f"profile={PROFILE}", f"experiment={PROFILE}", *OVERRIDES])
print("default backbone:", cfg.model.backbone, "(DINOv3 with HF_TOKEN, DINOv2 otherwise)")
print(OmegaConf.to_yaml(cfg.experiment))

## 2. Data

In [ ]:
from src.data import load_split
from src.synthetic import make_synthetic_data

if PROFILE == "smoke":
    data_dir = make_synthetic_data(ROOT / "data", ROOT / "runs" / "smoke_data")
else:
    data_dir = ROOT / cfg.data.dir
    if not (data_dir / "PETA" / "images").is_dir():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown<6", "tqdm"], check=True)
        subprocess.run([sys.executable, "download_datasets.py", "--data-dir", str(data_dir)], check=True)
cfg = with_values(cfg, {"data.dir": data_dir})

for name in ("train", "val"):
    split = load_split(data_dir, name)
    domains, counts = np.unique(split.domains, return_counts=True)
    missing = sum(not (data_dir / p).is_file() for p in split.images[:200])
    print(f"{name}: {len(split)} images, {len(split.queries)} queries, "
          f"{dict(zip(domains.tolist(), counts.tolist()))}, missing images (first 200): {missing}")

## 3. Score functions

Official validation ground truth plus Gaussian logit noise, no images needed.

In [ ]:
from src import evaluate

evaluate.run(with_values(cfg, {"data.dir": ROOT / "data", "eval.simulate": 3.0, "eval.max_queries": 300,
                               "eval.scores": ["l1", "loglik", "structured", "mix"], "eval.by_domain": False}));

## 4. Backbones

Pre-trained weights of every candidate backbone and the estimated inference time of a 28,000-image gallery on the current hardware.

In [ ]:
from src import benchmark

backbones = sorted({o.split("=", 1)[1] for opts in cfg.experiment.candidates.values()
                    for o in opts if o.startswith("model.backbone=")})
benchmark.run(with_values(cfg, {"benchmark.backbones": backbones, "benchmark.iters": 1, "benchmark.batch_size": 8}));

## 5. Fine-tuning the candidates, LODO, ensemble, WiSE-FT, export

`src.experiments` trains each candidate of `configs/experiment/<profile>.yaml`, scores it on the official validation split (calibration fitted on half of the queries, measured on the other half), adds an ensemble of the top-k and a WiSE-FT variant of the best one, runs LODO for the winner (`experiment.lodo`) and exports it.

In [ ]:
from src import experiments

summary = experiments.run(cfg)

## 6. LODO of the winner

Per fold metrics (train on two domains, calibrate on half of the third, evaluate on the other half), the score configuration selected on the mean over folds and the averaged calibration, all frozen into the winning checkpoint.

In [ ]:
import json

from src.checkpoint import Bundle

winner = Bundle.load(summary["winner_checkpoint"])
print("winner:", summary["winner"])
print("score:", winner.score)
lodo_summary = winner.meta.get("lodo")
print(json.dumps({k: lodo_summary[k] for k in ("best", "mean", "folds")}, indent=2) if lodo_summary
      else "no LODO for this winner (ensemble / WiSE-FT keep their validation calibration)")

## 7. Test-like evaluation of the winner

Frozen checkpoint (score, calibration, TTA) on a validation gallery whose queries are drawn by size like the hidden test; reports each source domain, seen vs novel queries, ECE and calibrated vs raw scores.

In [ ]:
resample = 0 if PROFILE == "smoke" else 367
evaluate.run(with_values(cfg, {"eval.checkpoint": summary["winner_checkpoint"], "eval.resample": resample}));

## 8. End-to-end inference with the exported submission

Imports the exported `run.py` (not the repository code) and calls `rank_gallery` like the ingestion program, then computes the official metrics.

In [ ]:
import importlib.util

from src.attributes import ATTRIBUTE_NAMES
from src.data import query_mask
from src.metrics import evaluate_output

submission_dir = Path(cfg.experiment.export)
spec = importlib.util.spec_from_file_location("submission_run", submission_dir / "run.py")
submission = importlib.util.module_from_spec(spec)
spec.loader.exec_module(submission)
submission.WORKERS = cfg.run.workers

gallery = load_split(data_dir, "val")
if len(gallery.queries) > 367:
    gallery = gallery.subset(query_mask(gallery, num_queries=367, by_size=True))
sample = {"attribute_names": list(ATTRIBUTE_NAMES), "queries": gallery.queries.tolist(),
          "gallery": [{"image_path": str(data_dir / p)} for p in gallery.images]}

t0 = time.time()
output = submission.rank_gallery(sample)
print(f"{len(gallery.queries)} queries x {len(gallery)} images in {time.time() - t0:.1f}s, "
      f"submission {summary['archive']}")
evaluate_output(output, gallery.queries, gallery.labels)

## Next steps

* Edit `configs/experiment/full.yaml` to add or change candidates; `experiment.lodo=all` reports LODO for every one of them.
* Command line equivalent of this notebook: `python -m src.experiments profile=full experiment=full`.
* Keep `score.transductive=false` until the organisers approve it in writing.